In [1]:

# Jupyter Dataset Loader UI
# provide a Jupyter UI for for exploring, loading, and visualizing various biosignals.

'''
Usage
-----
1. Open the notebook in VScode.
2. Run all cells.
3. Select a dataset from the dropdown.
4. Choose an example case or enter a manual case ID.
5. Select one feature, or enable "Plot all features".
6. Click Plot to visualize the signals.

'''

# Set the full dataset path here(only needed when check manual cases)
DATASET_DIR = "/Volumes/datasets/datasets"

In [1]:
# --- Jupyter UI for stress dataset loaders ---

from __future__ import annotations
import os, sys, json, contextlib, io, traceback
from typing import Dict, Any, List, Optional

import ipywidgets as w
from IPython.display import display, clear_output

# --- BEGIN: project-specific paths  ---
DATASET_LITE_DIR = "datasets_lite"
METADATA_DIR = "metadata"
# --- END: project-specific paths ---


from DatasetLoader.WFDBLoader import WFDBLoader
from DatasetLoader.EmpaticaE4Loader import EmpaticaE4Loader
from DatasetLoader.EDFLoader import EDFLoader
from DatasetLoader.PropofolLoader import PropofolLoader
from DatasetLoader.MHealthLoader import MHealthLoader
from DatasetLoader.CardioRespiratoryLoader import CardioRespiratoryLoader

LOADER_CLASSES = {
    "WFDBLoader": WFDBLoader,
    "EmpaticaE4Loader": EmpaticaE4Loader,
    "EDFLoader": EDFLoader,
    "PropofolLoader": PropofolLoader,
    "MHealthLoader": MHealthLoader,
    "CardioRespiratoryLoader": CardioRespiratoryLoader,
}

def die(msg: str):
    raise RuntimeError(msg)

def load_json_metadata(dataset: str) -> Dict[str, Any]:
    meta_path = os.path.join(METADATA_DIR, f"{dataset}.json")
    if not os.path.isfile(meta_path):
        die(f"Metadata JSON not found: {meta_path}")
    with open(meta_path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_data_path(base_folder: str, dataset: str, entry_point: str) -> str:
    base_root = os.path.abspath(os.path.join(base_folder, dataset))
    ep = entry_point if entry_point else "./"
    return os.path.abspath(os.path.join(base_root, ep))

def init_loader(loader_name: str, data_path: str, case: str):
    LoaderClass = LOADER_CLASSES.get(loader_name)
    if LoaderClass is None:
        die(f"Unknown loader '{loader_name}'. Allowed: {', '.join(LOADER_CLASSES.keys())}")

    if loader_name == "WFDBLoader":
        return LoaderClass(case, path=data_path)
    if loader_name == "EmpaticaE4Loader":
        return LoaderClass(case=case, path=data_path)
    if loader_name == "EDFLoader":
        return LoaderClass(case=case, path=data_path)
    if loader_name == "PropofolLoader":
        return LoaderClass(case, path=data_path)
    if loader_name == "MHealthLoader":
        return LoaderClass(case=case, path=data_path)
    if loader_name == "CardioRespiratoryLoader":
        if "_" in case:
            return LoaderClass(path=data_path, id_test=case)
        else:
            return LoaderClass(path=data_path, id=case)
    die(f"Unknown loader '{loader_name}'")

def list_datasets() -> List[str]:
    if not os.path.isdir(METADATA_DIR):
        return []
    return sorted([f[:-5] for f in os.listdir(METADATA_DIR) if f.endswith(".json")])

# -------- Widgets --------
ds_dropdown = w.Dropdown(
    options=["<select>"] + list_datasets(),
    value="<select>",
    description="1. Dataset:",
    layout=w.Layout(width="350px")
)

meta_out = w.Output(
    layout=w.Layout(
        border="1px solid #ddd",
        padding="8px",
        white_space="normal", # <-- allow automatic wrapping
        word_break="break-word", 
    )
)

case_mode = w.ToggleButtons(
    options=[("Example Case", "example"), ("Manual Case", "manual")],
    value="example",
    description="Case:",
)
case_text = w.Text(
    placeholder="Enter case ID (e.g., 1121 or S9)…",
    layout=w.Layout(width="300px"),
    disabled=True,
)

plot_all_chk = w.Checkbox(description="Plot all features", value=False)
features_ms = w.SelectMultiple(
    options=[],
    description="Features",
    rows=8,
    layout=w.Layout(width="320px"),
    disabled=False
)
refresh_feat_btn = w.Button(description="Refresh features", icon="refresh")
plot_btn = w.Button(description="Plot", button_style="primary", icon="play")
stdout_out = w.Output(layout=w.Layout(border="1px solid #ddd", padding="8px"))
status_html = w.HTML()

_current_meta: Optional[Dict[str, Any]] = None

def _render_meta(meta: Dict[str, Any]):
    with meta_out:
        meta_out.clear_output()
        print("=== Dataset Metadata Summary ===")
        print(f"Name           : {meta.get('name', 'N/A')}")
        print(f"Description    : {meta.get('notes', 'N/A')}")
        print(f"Data Loader    : {meta.get('loader', 'N/A')}")
        print(f"Record Count   : {meta.get('record_count', 'N/A')}")
        print(f"Format         : {meta.get('format', 'N/A')}")
        print(f"Example Case   : {meta.get('example_case', 'N/A')}")
        structure = meta.get("structure", {})
        if isinstance(structure, dict) and "sensor_features" in structure:
            feats = structure["sensor_features"]
            if isinstance(feats, list):
                print(f"Dataset Features: {', '.join(feats)}")
            else:
                print("Dataset Features: N/A")
        else:
            print("Dataset Features: N/A")

def _dataset_changed(change=None):
    global _current_meta
    ds = ds_dropdown.value
    features_ms.options = []
    if ds == "<select>":
        _current_meta = None
        meta_out.clear_output()
        status_html.value = ""
        case_text.value = ""
        case_text.disabled = True
        case_mode.value = "example"
        return

    try:
        _current_meta = load_json_metadata(ds)
        _render_meta(_current_meta)

        # Prefill example case if present
        ex = _current_meta.get("example_case")
        if ex:
            case_mode.value = "example"
            case_text.value = str(ex)
            case_text.disabled = True
        else:
            case_mode.value = "manual"
            case_text.disabled = False
            case_text.value = ""

        # Populate feature options from metadata if available
        feats = []
        structure = _current_meta.get("structure", {})
        if isinstance(structure, dict) and isinstance(structure.get("sensor_features"), list):
            feats = list(structure["sensor_features"])
        features_ms.options = sorted(feats)
        status_html.value = "<span style='color:green'>Loaded metadata.</span>"
    except Exception as e:
        meta_out.clear_output()
        status_html.value = f"<span style='color:red'>Failed to load metadata: {e}</span>"

def _case_mode_changed(change=None):
    if case_mode.value == "example":
        case_text.disabled = True
        if _current_meta:
            case_text.value = str(_current_meta.get("example_case", "")) or ""
    else:
        case_text.disabled = False

def _refresh_features_clicked(btn):
    """Try to load the loader and ask it for features if possible; else leave metadata-derived ones."""
    if not _current_meta:
        status_html.value = "<span style='color:red'>Select a dataset first.</span>"
        return
    ds = ds_dropdown.value
    loader_name = _current_meta.get("loader")
    if not loader_name:
        status_html.value = "<span style='color:red'>'loader' missing in metadata.</span>"
        return

    case = case_text.value.strip()
    if case_mode.value == "example":
        case = str(_current_meta.get("example_case", "") or "").strip()
    if not case:
        status_html.value = "<span style='color:red'>Provide a case ID (or switch to Example Case).</span>"
        return

    base_folder = DATASET_LITE_DIR if case_mode.value == "example" else DATASET_DIR
    entry_point = _current_meta.get("entry_point", "./")
    data_path = build_data_path(base_folder, ds, entry_point)

    try:
        loader = init_loader(loader_name, data_path, case)
        # Try common capability discovery methods:
        feats = None
        for attr in ("get_available_features", "list_features", "available_features"):
            if hasattr(loader, attr):
                v = getattr(loader, attr)
                feats = v() if callable(v) else v
                break
        if feats and isinstance(feats, (list, tuple)):
            features_ms.options = sorted([str(f) for f in feats])
            status_html.value = "<span style='color:green'>Features refreshed from loader.</span>"
        else:
            status_html.value = "<span>Loader does not expose a feature list; using metadata features (if any).</span>"
    except Exception as e:
        status_html.value = f"<span style='color:red'>Failed to refresh features: {e}</span>"

def _plot_clicked(btn):
    stdout_out.clear_output()
    if not _current_meta:
        status_html.value = "<span style='color:red'>Select a dataset first.</span>"
        return
    ds = ds_dropdown.value
    meta = _current_meta
    loader_name = meta.get("loader")
    if not loader_name:
        status_html.value = "<span style='color:red'>'loader' missing in metadata.</span>"
        return

    case = case_text.value.strip()
    if case_mode.value == "example":
        case = str(meta.get("example_case", "") or "").strip()
    if not case:
        status_html.value = "<span style='color:red'>Provide a case ID (or switch to Example Case).</span>"
        return

    base_folder = DATASET_LITE_DIR if case_mode.value == "example" else DATASET_DIR
    entry_point = meta.get("entry_point", "./")
    data_path = build_data_path(base_folder, ds, entry_point)

    try:
        loader = init_loader(loader_name, data_path, case)
        with stdout_out:
            # mimic your CLI's behavior: print summary if available
            if hasattr(loader, "print_summary"):
                # capture stdout from print_summary
                f = io.StringIO()
                with contextlib.redirect_stdout(f):
                    loader.print_summary()
                print(f.getvalue())
            # plot
            if plot_all_chk.value:
                if not hasattr(loader, "plot_all"):
                    raise RuntimeError(f"Loader '{loader_name}' has no plot_all() method.")
                loader.plot_all()
            else:
                feats = list(features_ms.value) if features_ms.value else []
                if not feats:
                    raise RuntimeError("Select at least one feature or check 'Plot all features'.")
                if not hasattr(loader, "plot"):
                    raise RuntimeError(f"Loader '{loader_name}' has no plot() method.")
                loader.plot(feats)
        status_html.value = "<span style='color:green'>Done.</span>"
    except Exception as e:
        tb = traceback.format_exc(limit=2)
        status_html.value = f"<span style='color:red'>Plot failed: {e}</span>"
        with stdout_out:
            print(tb)

# --- Wire events ---
ds_dropdown.observe(_dataset_changed, names="value")
case_mode.observe(_case_mode_changed, names="value")
refresh_feat_btn.on_click(_refresh_features_clicked)
plot_btn.on_click(_plot_clicked)

# --- Initial render ---
controls_row1 = w.HBox([ds_dropdown, case_mode, case_text])
controls_row2 = w.HBox([plot_all_chk, refresh_feat_btn, plot_btn])

# Metadata at top
metadata_col = w.VBox([w.HTML("<b>2. Metadata</b>"), meta_out, status_html])
# Features below metadata
features_col = w.VBox([w.HTML("<b>3. Features</b>"), features_ms, controls_row2])

ui = w.VBox([
    controls_row1,
    metadata_col,
    features_col,
    w.HTML("<b>4. Output</b>"),
    stdout_out,
])

display(ui)
